# 07 - Held-out Error Analysis

This notebook uses the saved model and the same untouched, stratified 20% test split as notebook 06. It does not fit, calibrate, tune, or modify the model. The real label is `f` (0) and the fraud label is `t` (1).

## Load the model and reconstruct the exact test split

The raw and processed rows are checked for matching text and labels before the original `random_state=42` split is recreated.

In [4]:
from pathlib import Path
import json
import re
from collections import Counter
from html import unescape

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_PATH = PROJECT_DIR / "models" / "fake_job_detector.joblib"
METADATA_PATH = PROJECT_DIR / "models" / "fake_job_detector_metadata.json"
PROCESSED_PATH = PROJECT_DIR / "data" / "processed" / "processed_job_postings.csv"
RAW_PATH = PROJECT_DIR / "data" / "raw" / "DataSet.csv"
REPORT_PATH = PROJECT_DIR / "report" / "error_analysis_report.md"

for path in [MODEL_PATH, METADATA_PATH, PROCESSED_PATH, RAW_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

model = joblib.load(MODEL_PATH)
with METADATA_PATH.open(encoding="utf-8") as file:
    model_metadata = json.load(file)
numeric_features = model_metadata["numeric_features"]
categorical_features = model_metadata["categorical_features"]
text_feature = model_metadata["text_feature"]

processed = pd.read_csv(
    PROCESSED_PATH,
    usecols=[text_feature, "fraudulent", *numeric_features],
    low_memory=False,
)
raw = pd.read_csv(RAW_PATH, low_memory=False).drop_duplicates().copy().reset_index(drop=True)
if len(raw) != len(processed):
    raise ValueError(f"Raw and processed row counts differ: {len(raw)} vs {len(processed)}")

text_columns = [c for c in ["title", "company_profile", "description", "requirements", "benefits"] if c in raw]
needed_raw_columns = [c for c in categorical_features if c in raw.columns]
if len(needed_raw_columns) != len(categorical_features):
    raise ValueError("A categorical model input column is missing from the raw dataset.")
raw_view = raw[[*needed_raw_columns, "fraudulent", *text_columns]].copy()
for column in raw_view.select_dtypes(include=["object", "string"]).columns:
    raw_view[column] = raw_view[column].replace(r"^\s*$", pd.NA, regex=True)

def clean_text(value):
    text = "" if pd.isna(value) else str(value)
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\b(?:https?://|www\.)\S+", " URL ", text, flags=re.IGNORECASE)
    text = re.sub(r"\b[\w.+%-]+@[\w.-]+\.[a-z]{2,}\b", " EMAIL ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

rebuilt_text = raw_view[text_columns].fillna("").astype(str).agg(" ".join, axis=1).map(clean_text).to_numpy()
if not np.array_equal(rebuilt_text, processed[text_feature].fillna("").astype(str).to_numpy()):
    raise ValueError("Raw and processed combined text are not aligned row by row.")
labels = raw_view["fraudulent"].astype(str).str.strip().str.lower().to_numpy()
if not np.array_equal(labels, processed["fraudulent"].astype(str).str.strip().str.lower().to_numpy()):
    raise ValueError("Raw and processed target labels are not aligned row by row.")
if set(labels) != {"f", "t"}:
    raise ValueError(f"Unexpected target labels: {sorted(set(labels))}")

X = processed[[text_feature, *numeric_features]].copy()
for column in categorical_features:
    X[column] = raw_view[column]
X = X[[text_feature, *categorical_features, *numeric_features]]
y = pd.Series(labels, index=X.index).map({"f": 0, "t": 1}).astype("int8")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
test_raw = raw_view.iloc[X_test.index].copy()
y_pred = model.predict(X_test).astype(int)
fraud_probability = model.predict_proba(X_test)[:, list(model.classes_).index(1)]

print(f"Model: {type(model).__name__}; test examples: {len(X_test):,}")
print("Exact row/text/label alignment verified; model was not fit in this notebook.")

Model: CalibratedClassifierCV; test examples: 3,529
Exact row/text/label alignment verified; model was not fit in this notebook.


## Separate correct predictions and errors

True positive means a fraudulent posting correctly flagged. True negative means a legitimate posting correctly cleared. False positive is a legitimate posting incorrectly flagged; false negative is a fraudulent posting missed.

In [5]:
test_results = test_raw.copy()
test_results["actual"] = y_test.to_numpy()
test_results["predicted"] = y_pred
test_results["fraud_probability"] = fraud_probability

groups = {
    "True positives": test_results[(test_results.actual == 1) & (test_results.predicted == 1)],
    "True negatives": test_results[(test_results.actual == 0) & (test_results.predicted == 0)],
    "False positives": test_results[(test_results.actual == 0) & (test_results.predicted == 1)],
    "False negatives": test_results[(test_results.actual == 1) & (test_results.predicted == 0)],
}
counts = pd.DataFrame({"prediction_group": list(groups), "count": [len(frame) for frame in groups.values()]})
display(counts)
print("Total check:", sum(len(frame) for frame in groups.values()), "of", len(test_results))

,prediction_group,count
0,True positives,136
1,True negatives,3336
2,False positives,21
3,False negatives,36


Total check: 3529 of 3529


## False positives and false negatives

At most 10 examples of each type are shown (all available examples are shown when a category contains fewer than 10). Long text is kept intact for inspection.

In [14]:
def example_table(frame):
    columns = [c for c in ["title", "company_profile", "description"] if c in frame.columns]
    table = frame[columns + ["actual", "predicted", "fraud_probability"]].copy()
    for column in ["title", "company_profile", "description"]:
        if column in table:
            table[column] = table[column].fillna("").astype(str).str.replace(r"[\r\n]+", " ", regex=True).str.replace("|", "\\|", regex=False)
    table["actual"] = table["actual"].map({0: "Legitimate (f)", 1: "Fraudulent (t)"})
    table["predicted"] = table["predicted"].map({0: "Legitimate (f)", 1: "Fraudulent (t)"})
    table["fraud_probability"] = table["fraud_probability"].map(lambda value: f"{value:.2%}")
    return table

false_positives = groups["False positives"]
false_negatives = groups["False negatives"]
print(f"False positives ({len(false_positives)} total)")
display(example_table(false_positives.head(10)))
print(f"False negatives ({len(false_negatives)} total)")
display(example_table(false_negatives.head(10)))

False positives (21 total)


,title,company_profile,description,actual,predicted,fraud_probability
5708,Part Time Ground representative,,<p>Work for a world leading NGO- Non Governmen...,Legitimate (f),Fraudulent (t),77.57%
12330,Surveyors / Order Takers,,<p>Get paid $12 per hour to walk 3 miles per d...,Legitimate (f),Fraudulent (t),50.75%
11806,Data Entry ($18/hr),<p>Monroe Staffing is a full-service staffing ...,<p>Data entry clerk needed for a 16 week proje...,Legitimate (f),Fraudulent (t),62.65%
14944,Payroll Processor,,<p>JOB DESCRIPTION<br>Classification: Payroll ...,Legitimate (f),Fraudulent (t),88.59%
14150,Office Manager,<p>Mousera is accelerating medical advancement...,<p><b>NOTE: We are looking for someone who has...,Legitimate (f),Fraudulent (t),83.23%
14070,Java Developer,,<ul> <li>Minimum 7 years experience in <b>Java...,Legitimate (f),Fraudulent (t),61.02%
14485,Payroll Clerk $20/hr,,<p>The Accounting clerk will work in a team en...,Legitimate (f),Fraudulent (t),72.39%
10356,Medical Receptionist/Scheduler,,<p><b><b>ReGenesis Counseling &amp; Assessment...,Legitimate (f),Fraudulent (t),72.37%
16363,Administrative Assistant,,<p>We currently have an opening for an Adminis...,Legitimate (f),Fraudulent (t),70.06%
10648,Maintenance Mechanic,,<p>Principal Responsibilities:</p> <p>1. Maint...,Legitimate (f),Fraudulent (t),72.26%


False negatives (36 total)


,title,company_profile,description,actual,predicted,fraud_probability
2917,"LEAN Program Manager, Carol Stream, IL",,<p>The LEAN Programs Manager will lead strateg...,Fraudulent (t),Legitimate (f),5.31%
17546,Junior Project Manager,,<p>Descripción de prueba</p>,Fraudulent (t),Legitimate (f),0.77%
8787,Maintenance Specialist,<p>As the industry’s largest supply contractin...,<p>Responsibilities<br>Responsibilities:<br>Re...,Fraudulent (t),Legitimate (f),48.52%
17160,Customer Service,,"<p>Smart, creative, personable Customer Servic...",Fraudulent (t),Legitimate (f),48.30%
17451,"Engineering Senior Supervisor, Livonia, MI",,<p>Responsible for leading a small team of eng...,Fraudulent (t),Legitimate (f),14.39%
17455,User Experience Designer \| Mobile,,<p>We are looking for UX Designer to help us c...,Fraudulent (t),Legitimate (f),7.29%
16326,Nursing & Therapy Positions- Home Care,,<p><b>Nursing &amp; Therapy Positions-Home Car...,Fraudulent (t),Legitimate (f),5.25%
5144,Final Expense Agent,,<p><!--[if gte mso 9]><xml> <o:OfficeDocument...,Fraudulent (t),Legitimate (f),5.38%
17524,Personal Assistant,,<p>Job Description</p> <p><br>Vertex Global So...,Fraudulent (t),Legitimate (f),15.56%
6335,Integration Engineer with HL7 & EMR.,,"<p><b> </b></p> <p>Location: Plano, TX</p> <p>...",Fraudulent (t),Legitimate (f),4.80%


## Error patterns and false-negative checks

The summaries below are descriptive. A short description is defined here as fewer than 250 characters. A salary value is called uncommon if its category occurs fewer than 10 times in the training partition; this is a transparent diagnostic threshold, not a trained rule.

In [9]:
from html import unescape
def is_missing(series):
    return series.isna() | series.astype("string").str.strip().eq("").fillna(False)

train_salary_counts = (
    test_raw[categorical_features].iloc[0:0]
    if "salary_range" not in raw_view.columns
    else raw_view.loc[X_train.index, "salary_range"].fillna("<MISSING>").astype(str).value_counts()
)

def error_summary(frame):
    summary = {"examples": len(frame)}
    for column, metric in [("company_profile", "missing_company_profile"), ("requirements", "missing_requirements")]:
        if column in frame:
            summary[metric] = int(is_missing(frame[column]).sum())
            summary[f"{metric}_percent"] = float(is_missing(frame[column]).mean() * 100) if len(frame) else np.nan
    if "description" in frame:
        description_lengths = frame["description"].fillna("").astype(str).str.len()
        summary["short_description_under_250_chars"] = int((description_lengths < 250).sum())
        summary["median_description_characters"] = float(description_lengths.median()) if len(frame) else np.nan
    if "salary_range" in frame:
        salaries = frame["salary_range"].fillna("<MISSING>").astype(str)
        summary["missing_salary_information"] = int(salaries.isin(["<MISSING>", "nan", "None", ""]).sum())
        summary["salary_categories_seen"] = int(salaries.nunique())
        summary["uncommon_salary_under_10_training_rows"] = int(
            salaries.map(lambda value: train_salary_counts.get(value, 0) < 10).sum()
        )
    return summary

error_pattern_table = pd.DataFrame(
    {name: error_summary(groups[name]) for name in ["False positives", "False negatives"]}
).T
display(error_pattern_table)

def top_terms(frame, n=20):
    if frame.empty:
        return pd.DataFrame(columns=["term_or_phrase", "count"])
    text_fields = [c for c in ["title", "company_profile", "description", "requirements", "benefits"] if c in frame]
    parts = []
    for value in frame[text_fields].fillna("").astype(str).values.ravel():
        cleaned = unescape(value).lower()
        cleaned = re.sub(r"<!--[\s\S]*?-->", " ", cleaned)
        cleaned = re.sub(r"<[^>]+>", " ", cleaned)
        parts.append(cleaned)
    tokens = re.findall(r"[a-z][a-z']{2,}", " ".join(parts))
    stop_words = {
        "the", "and", "for", "with", "you", "your", "our", "are", "will", "this", "that", "from",
        "have", "has", "was", "were", "who", "what", "when", "where", "how", "all", "any", "not",
        "job", "work", "role", "team", "company", "business", "experience", "skills", "required",
        "must", "can", "able", "candidate", "position", "including", "about", "their", "them",
        "false", "true", "name", "priority", "accent", "medium", "locked", "lsdexception",
        "unhidewhenused", "semihidden", "font", "color", "xml", "style", "val", "xmlns",
    }
    tokens = [token for token in tokens if token not in stop_words and not token.startswith("lsd")]
    unigram_counts = Counter(tokens)
    bigram_counts = Counter(f"{left} {right}" for left, right in zip(tokens, tokens[1:]))
    counts = unigram_counts + bigram_counts
    return pd.DataFrame(counts.most_common(n), columns=["term_or_phrase", "count"])

print("Most common words and adjacent phrases in false-negative text:")
false_negative_terms = top_terms(false_negatives)
display(false_negative_terms)

,examples,missing_company_profile,missing_company_profile_percent,missing_requirements,missing_requirements_percent,short_description_under_250_chars,median_description_characters,missing_salary_information,salary_categories_seen,uncommon_salary_under_10_training_rows
False positives,21.0,16.0,76.190476,2.0,9.52381,3.0,977.0,12.0,10.0,8.0
False negatives,36.0,31.0,86.111111,9.0,25.00000,7.0,922.0,22.0,15.0,11.0


Most common words and adjacent phrases in false-negative text:


,term_or_phrase,count
0,time,31
1,data,30
2,management,27
3,url,27
4,customer,26
5,sales,23
6,lean,22
7,information,22
8,ability,21
9,products,20


## Practical feature-engineering ideas and report

Use the observed error counts and false-negative terms as hypotheses for future experiments. Any new feature or threshold must be evaluated using training-only validation; do not tune against this test set.

In [15]:
def markdown_table(frame):
    if frame.empty:
        return "No examples."
    columns = [str(column) for column in frame.columns]
    lines = ["| " + " | ".join(columns) + " |", "|" + "|".join(["---"] * len(columns)) + "|"]
    for values in frame.itertuples(index=False, name=None):
        cells = [str(value).replace("|", "\\|").replace("\n", " ") for value in values]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

fp_summary = error_summary(false_positives)
fn_summary = error_summary(false_negatives)
fp_terms = top_terms(false_positives, 12)
top_fn_markdown = markdown_table(false_negative_terms) if len(false_negative_terms) else "No false negatives were available."
fp_term_text = ", ".join(fp_terms["term_or_phrase"].head(8).tolist()) if len(fp_terms) else "not available"
top_false_positives = markdown_table(example_table(false_positives.head(10)))
top_false_negatives = markdown_table(example_table(false_negatives.head(10)))
pattern_text = (
    "False-positive text most often included these terms/phrases: " + fp_term_text
    + ". These overlaps are descriptive, not proof of why the classifier made each decision. "
    + f"False negatives were frequently missing company profiles ({fn_summary.get('missing_company_profile', 0)}/{len(false_negatives)}) and salary information ({fn_summary.get('missing_salary_information', 0)}/{len(false_negatives)}); {fn_summary.get('missing_requirements', 0)} lacked requirements and {fn_summary.get('short_description_under_250_chars', 0)} had descriptions under 250 characters."
)
false_negative_summary_lines = [
    f"- Company profile missing: {fn_summary.get('missing_company_profile', 0)} / {len(false_negatives)} ({fn_summary.get('missing_company_profile_percent', 0):.1f}%).",
    f"- Requirements missing: {fn_summary.get('missing_requirements', 0)} / {len(false_negatives)} ({fn_summary.get('missing_requirements_percent', 0):.1f}%).",
    f"- Description under 250 characters: {fn_summary.get('short_description_under_250_chars', 0)} / {len(false_negatives)}.",
    f"- Missing salary information: {fn_summary.get('missing_salary_information', 0)} / {len(false_negatives)}.",
    f"- Salary values occurring fewer than 10 times in training: {fn_summary.get('uncommon_salary_under_10_training_rows', 0)} / {len(false_negatives)}.",
]
feature_ideas = [
    "Add section-aware text indicators for suspicious claims or requests, scored separately in the description, requirements, and company profile (for example payment/fee requests and messaging-only contact cues). Validate phrase patterns on training-only validation data.",
    "Add robust compensation features: salary present/missing, parseable lower/upper bounds, currency, hourly-versus-annual clues, and outlier or unusually broad range flags; treat unparsable values as unknown rather than assuming fraud.",
    "Add structured completeness and consistency signals, such as company-profile and requirements missingness, description length, and agreement between remote-work wording and telecommuting metadata. Evaluate all signals without using the held-out test set for selection.",
]
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
report_lines = [
    "# Held-out Error Analysis Report", "", "## Method", "",
    f"Loaded {MODEL_PATH.relative_to(PROJECT_DIR)} and reconstructed the untouched stratified 80/20 split using random_state=42. Raw and processed combined text and labels were checked row by row. The model was only used for prediction; this notebook did not retrain, calibrate, tune, or alter the model or test set. Test rows: {len(test_results):,}.", "",
    "## Prediction groups", "",
    "| Group | Definition | Count |", "|---|---|---:|",
    f"| True positives | Fraud correctly predicted as fraud | {len(groups['True positives'])} |",
    f"| True negatives | Legitimate correctly predicted as legitimate | {len(groups['True negatives'])} |",
    f"| False positives | Legitimate predicted as fraud | {len(false_positives)} |",
    f"| False negatives | Fraud predicted as legitimate | {len(false_negatives)} |", "",
    "## False positives (up to 10)", "", top_false_positives, "",
    "## False negatives (up to 10)", "", top_false_negatives, "",
    "## Pattern observations", "", pattern_text, "",
    "### False-negative checks", "", *false_negative_summary_lines, "",
    "### Common false-negative words and phrases", "", top_fn_markdown, "",
    "Word and adjacent two-word phrase counts are frequency summaries of false-negative text after HTML tags and entities are normalized. They are descriptive, not causal explanations, and may be influenced by repeated terms.", "",
    "## Three practical feature-engineering improvements", "",
    f"1. {feature_ideas[0]}", f"2. {feature_ideas[1]}", f"3. {feature_ideas[2]}", "",
    "Any feature experiments should be compared using training-only validation. Do not select features, change thresholds, or tune on the held-out test results in this report.",
]
report = "\n".join(report_lines)
REPORT_PATH.write_text(report, encoding="utf-8")
print(f"Saved error analysis report: {REPORT_PATH}")
print("No fitting or model changes were performed.")

Saved error analysis report: <PROJECT_ROOT>\report\error_analysis_report.md
No fitting or model changes were performed.
